# 02 - Dataset Construction

## Objective

The objective of this notebook is to transform the raw Polymarket data collected in `01_Data_Acquisition.ipynb` into a clean dataset.

The raw data consists of two main components:

1. Historical implied probability time series.
2. Market-level metadata.

The goal is to combine both sources into a structured dataset suitable for exploratory analysis, calibration studies, Bayesian probability estimation, and future signal generation.

---

## Inputs

This notebook uses the following datasets generated in the previous phase:

```text
data/raw/prices_raw.csv
data/raw/markets_metadata.csv
```

--- 

### prices_raw.csv

Contains historical implied probabilities for each market.

fields

| Column                | Description                                          |
| --------------------- | ---------------------------------------------------- |
| `timestamp`           | Timestamp of the observed market-implied probability |
| `implied_probability` | YES contract price interpreted as probability        |
| `market_id`           | Unique Polymarket market identifier                  |
| `question`            | Market question                                      |
| `slug`                | Market slug                                          |

 

### markets_metadata.csv

Contains market-level information.

fields:

| Column          | Description                         |
| --------------- | ----------------------------------- |
| `id`            | Unique Polymarket market identifier |
| `question`      | Market question                     |
| `endDate`       | Market resolution or closing date   |
| `volume`        | Total traded volume                 |
| `liquidity`     | Market liquidity                    |
| `outcomes`      | Possible market outcomes            |
| `outcomePrices` | Final outcome prices                |
| `yes_token_id`  | YES outcome CLOB token identifier   |


--- 


## Dataset Construction Logic

The construction process follows four steps:

1. Load raw historical probabilities and market metadata.
2. Validate data types, timestamps, and identifiers.
3. Aggregate historical probability series at the market level.
4. Merge market-level probability features with metadata and final outcomes.


The final output of this notebook will be a market-level dataset where each row represents one resolved prediction market.

structure:

| market_id | question | start_date | end_date | final_probability | mean_probability | probability_volatility | volume | liquidity | outcome |
| --------- | -------- | ---------- | -------- | ----------------- | ---------------- | ---------------------- | ------ | --------- | ------- |


---

This dataset will be used in the next notebooks for:

1. Exploratory analysis
2. Calibration analysis
3. Brier Score evaluation
4. Bayesian fair probability estimation
5. Mispricing detection
6. Backtesting

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
# File paths
PRICES_PATH = "../data/raw/prices_raw.csv"
METADATA_PATH = "../data/raw/markets_metadata.csv"
# Load raw datasets
df_prices = pd.read_csv(PRICES_PATH)
df_metadata = pd.read_csv(METADATA_PATH)
display(df_prices.head(2))
display(df_metadata.head(2))

,timestamp,implied_probability,market_id,question,slug
0,2023-02-07 01:00:32+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
1,2023-02-07 02:00:53+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...


,id,question,slug,category,endDate,closed,volume,liquidity,outcomes,outcomePrices,clobTokenIds,yes_token_id,endDate_dt
0,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,NaN,2023-05-01T00:00:00Z,True,35102.732603,25.00501,"[""Yes"", ""No""]","[""0"", ""1""]","[""38374278491791697742783566758802364186445803...",3837427849179169774278356675880236418644580318...,2023-05-01 00:00:00+00:00
1,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,NaN,2023-04-02T00:00:00Z,True,500.000000,0.00000,"[""Reds"", ""Pirates""]","[""1"", ""0""]","[""80040235635822024858376428020052595519279817...",8004023563582202485837642802005259551927981755...,2023-04-02 00:00:00+00:00


## Basic Data Validation

In [2]:
print(df_prices.info())

print("\n")
print(df_metadata.info())

print("\n")
display(df_prices.isna().sum())

print("\n")
display(df_metadata.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 100642 entries, 0 to 100641
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   timestamp            100642 non-null  str    
 1   implied_probability  100642 non-null  float64
 2   market_id            100642 non-null  int64  
 3   question             100642 non-null  str    
 4   slug                 100642 non-null  str    
dtypes: float64(1), int64(1), str(3)
memory usage: 3.8 MB
None


<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             43 non-null     int64  
 1   question       43 non-null     str    
 2   slug           43 non-null     str    
 3   category       0 non-null      float64
 4   endDate        43 non-null     str    
 5   closed         43 non-null     bool   
 6   volume         43 non-null    

timestamp              0
implied_probability    0
market_id              0
question               0
slug                   0
dtype: int64

id                0
question          0
slug              0
category         43
endDate           0
closed            0
volume            0
liquidity        18
outcomes          0
outcomePrices     0
clobTokenIds      0
yes_token_id      0
endDate_dt        0
dtype: int64

In [3]:
print("unique markets in prices:")
print(df_prices["market_id"].nunique())
print("\nrows in metadata:")
print(len(df_metadata))
print("\nduplicate market:")
print(df_metadata["id"].duplicated().sum())

unique markets in prices:
43

rows in metadata:
43

duplicate market:
0


In [4]:
df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"],utc=True)
print("min timestamp:",df_prices["timestamp"].min())
print("max timestamp:",df_prices["timestamp"].max())

min timestamp: 2023-02-07 01:00:32+00:00
max timestamp: 2024-12-30 23:00:04+00:00
